In [2]:
'''
QuickSearch_V2_Res
Uses ResolutionHandling/processed_candidates as input dataset.
Writes outputs inside ResolutionHandling.
'''
import time
import numpy as np
import glob, os, sys
import pandas as pd
from pathlib import Path
import multiprocessing
import json, argparse

BASE = Path('/home/msp25gd/ResearchProjectMSc/ResolutionHandling')
SEARCH_BASE = Path('/home/msp25gd/ResearchProjectMSc/HR/search')
PROCESSED_DATASET = BASE / 'processed_candidates'
DEFAULT_PARAM = SEARCH_BASE / 'param.json'
DEFAULT_OUTDIR = BASE / 'QuickSearch_V2'

# Match original quicksearch_V2 execution context for relative %run imports inside asset.ipynb.
%cd /home/msp25gd/ResearchProjectMSc/HR/search
%run ./asset.ipynb
%run ./width_depth.ipynb

def _clean_token(x):
    if x is None:
        return None
    s = str(x).strip().lower()
    if not s:
        return None
    return s.replace(' ', '').replace('_', '').replace('-', '')


def dedup_candidates_by_v2_groups(cands, full_v2_path):
    """Deduplicate candidate names by merged V2 group IDs from full_metadata_V2.pkl."""
    if len(cands) == 0:
        return cands

    meta_v2 = pd.read_pickle(full_v2_path)
    name_cols = [c for c in ['Reduced', 'Sanitised', 'OBJECT', 'Object'] if c in meta_v2.columns]

    # token -> group id lookup
    lookup = {}
    for _, row in meta_v2.iterrows():
        gid = int(row['New Groups'])
        for c in name_cols:
            key = _clean_token(row[c])
            if key is not None and key not in lookup:
                lookup[key] = gid

    out = []
    seen_groups = set()
    unresolved = []

    for raw in cands.tolist():
        key = _clean_token(raw)
        gid = lookup.get(key)

        # Fallback through normalization regex rules loaded from grouping_V2 notebook logic
        if gid is None and key is not None and 'normalize_reduced' in globals():
            gid = lookup.get(_clean_token(normalize_reduced(key)))

        if gid is None:
            # Keep unresolved names to avoid accidental data loss
            unresolved.append(raw)
            out.append(raw)
            continue

        if gid in seen_groups:
            continue

        seen_groups.add(gid)
        out.append(raw)

    return np.array(out, dtype=object)


if __name__ == '__main__':
    startTime = time.time()

    parser = argparse.ArgumentParser(description='How to use QuickSearch_V2_Res', epilog='ASSET by RBW')
    parser.add_argument('--p', metavar='param.json', default=str(DEFAULT_PARAM), help='parameter file name')
    parser.add_argument('--nice', metavar='niceness', default=15, help='niceness of job (default: 15)')
    parser.add_argument('--line', metavar='line', default='K', help='define what atomic line to use (default: K)')
    parser.add_argument('--r', metavar='res-path', default=str(DEFAULT_OUTDIR) + '/', help='output path inside ResolutionHandling')
    parser.add_argument('--tag', metavar='tag', default='V2_RES', help='suffix tag for output filename (default: V2_RES)')
    parser.add_argument('--groups-meta', default='/home/msp25gd/Downloads/res/meta/full_metadata_V2.pkl',
                        help='path to merged V2 full metadata')
    parser.add_argument('--no-group-dedup', action='store_true',
                        help='disable deduplication by merged V2 groups')
    parser.add_argument('-f', '--f', dest='connection_file', default=None,
                        help='Jupyter kernel connection file')

    args = parser.parse_args()

    os.nice(int(args.nice))

    with open(args.p) as paramfile:
        param = json.load(paramfile)

    # Force the processed, per-group-uniform dataset built in ResolutionHandling.
    param['dataset'] = str(PROCESSED_DATASET) + '/'

    if not PROCESSED_DATASET.exists():
        raise FileNotFoundError(f'Processed dataset not found: {PROCESSED_DATASET}')

    print('Dataset root:', param['dataset'])
    all_paths = glob.glob(param['dataset'] + '*/')

    min_spectra = int(param.get('min_spectra', 1))
    # Require a minimum number of spectra so each target has enough epochs to form a robust group reference.
    dataset_path = [p for p in all_paths if np.load(p + 'spec/sK.npy').shape[0] >= min_spectra]
    print('Stars with >= {} spectra: {}/{}'.format(min_spectra, len(dataset_path), len(all_paths)))

    Search = ASSET(parameters=param, line=args.line)

    def quicksearch_with_rv_filter(star):
        Search.ccf = False
        spec_param = Search.spec_analysis(star)
        if spec_param is None:
            return None

        new_spectra, med, med_err = spec_param

        if Search.ccf:
            rv_shift = Search.X_corr(med.copy())
            cond100 = (Search.radial_velocity > rv_shift - 50) & (Search.radial_velocity < rv_shift + 50)

        dip_detected = False
        max_peak_seen = -np.inf

        for i in range(len(new_spectra)):
            spec = new_spectra[i]

            snr = Search.snr(spec, med, Search.spectra_err[i], med_err)
            sd = np.std(snr)
            if not np.isfinite(sd) or sd == 0:
                continue

            corr_snr = snr.copy()
            if Search.ccf:
                corr_snr[cond100] = np.nan
            corr_snr = corr_snr[Search.snr_idxrange]

            sig = corr_snr / sd

            min_detect = np.nanmin(sig)
            max_detect = np.nanmax(sig)
            if np.isfinite(max_detect):
                max_peak_seen = max(max_peak_seen, float(max_detect))

            if min_detect < Search.threshold:
                width = Search.get_width(sig)
                if width >= Search.width_filter:
                    dip_detected = True

        if dip_detected:
            if not np.isfinite(max_peak_seen):
                max_peak_seen = np.nan
            return (str(Search.target_red), float(max_peak_seen))

        return None

    with multiprocessing.Pool(param['cores']) as pool:
        out = pool.map(quicksearch_with_rv_filter, dataset_path)

    records = [item for item in out if item is not None]
    print('------------------')

    cands_raw = np.array([item[0] for item in records], dtype=object)
    peak_raw = np.array([item[1] for item in records], dtype=float)
    print('Raw candidates:', len(cands_raw))

    if args.no_group_dedup:
        cands = cands_raw
        print('Group dedup disabled')
    else:
        cands = dedup_candidates_by_v2_groups(cands_raw, args.groups_meta)
        print('After V2 group dedup:', len(cands), '(reduced by {})'.format(len(cands_raw) - len(cands)))

    # Map deduplicated names to peak values from the raw list
    peak_lookup = {}
    for name, peak in zip(cands_raw.tolist(), peak_raw.tolist()):
        if (name not in peak_lookup) or (np.isfinite(peak) and peak > peak_lookup[name]):
            peak_lookup[name] = peak

    cands_peak = np.array([peak_lookup.get(name, np.nan) for name in cands.tolist()], dtype=float)
    peak_threshold = abs(float(Search.threshold))

    # Split dip detections with significant positive peaks into requested bins
    bin_3p5_4_mask = (cands_peak >= peak_threshold) & (cands_peak < 4.0)
    bin_4_5_mask = (cands_peak >= 4.0) & (cands_peak < 5.0)
    bin_5_plus_mask = cands_peak >= 5.0

    peak_3p5_4 = cands[bin_3p5_4_mask]
    peak_4_5 = cands[bin_4_5_mask]
    peak_5_plus = cands[bin_5_plus_mask]

    print('Peak bins among candidates:')
    print('  +{:.1f} to +4 sigma: {}'.format(peak_threshold, len(peak_3p5_4)))
    print('  +4 to +5 sigma: {}'.format(len(peak_4_5)))
    print('  +5 sigma and above: {}'.format(len(peak_5_plus)))

    tag = args.tag.strip()
    if tag:
        cands_file = 'candidates_{}sig_{}cut_{}width_{}.npy'.format(Search.threshold, Search.cutoff, Search.width_filter, tag)
        peaks_file = 'candidate_max_peak_{}sig_{}cut_{}width_{}.npy'.format(Search.threshold, Search.cutoff, Search.width_filter, tag)
    else:
        cands_file = 'candidates_{}sig_{}cut_{}width.npy'.format(Search.threshold, Search.cutoff, Search.width_filter)
        peaks_file = 'candidate_max_peak_{}sig_{}cut_{}width.npy'.format(Search.threshold, Search.cutoff, Search.width_filter)

    if not os.path.exists(args.r):
        os.makedirs(args.r)
        print('new directory {} created!'.format(args.r))

    np.save(args.r + cands_file, cands)
    np.save(args.r + peaks_file, cands_peak)

    peak_bin_dirs = {
        'peak_3p5_to_4': peak_3p5_4,
        'peak_4_to_5': peak_4_5,
        'peak_5_plus': peak_5_plus,
    }

    for folder, arr in peak_bin_dirs.items():
        out_dir = os.path.join(args.r, folder)
        os.makedirs(out_dir, exist_ok=True)
        np.save(os.path.join(out_dir, 'targets.npy'), arr)

    executionTime = (time.time() - startTime)
    print('Execution time in seconds: ' + str(executionTime))

/home/msp25gd/ResearchProjectMSc/HR/search
Dataset root: /home/msp25gd/ResearchProjectMSc/ResolutionHandling/processed_candidates/
Stars with >= 3 spectra: 497/497
------------------
Raw candidates: 492
After V2 group dedup: 492 (reduced by 0)
Peak bins among candidates:
  +3.5 to +4 sigma: 125
  +4 to +5 sigma: 129
  +5 sigma and above: 118
new directory /home/msp25gd/ResearchProjectMSc/ResolutionHandling/QuickSearch_V2/ created!
Execution time in seconds: 144.7477114200592


In [3]:
print('QuickSearch_V2_Res candidates:', len(cands))
print('Output file:', cands_file)
print('Output directory:', args.r)

QuickSearch_V2_Res candidates: 492
Output file: candidates_-3.5sig_1.5cut_2width_V2_RES.npy
Output directory: /home/msp25gd/ResearchProjectMSc/ResolutionHandling/QuickSearch_V2/


In [4]:
# Diagnose merge-aware dedup impact on current candidate list
import os

def _norm_name_for_merge(name):
    # reuse notebook normalization if available
    if 'normalize_reduced' in globals():
        return normalize_reduced(str(name).strip().lower())
    return str(name).strip().lower()

cands_list = [str(x) for x in cands.tolist()]
norm = [_norm_name_for_merge(x) for x in cands_list]

print('Raw candidates:', len(cands_list))
print('Unique by normalized name:', len(set(norm)))
print('Potential reductions:', len(cands_list) - len(set(norm)))

# show collisions (names that collapse to same normalized key)
from collections import defaultdict
bucket = defaultdict(list)
for raw, n in zip(cands_list, norm):
    bucket[n].append(raw)
collisions = {k:v for k,v in bucket.items() if len(set(v)) > 1}
print('Colliding keys:', len(collisions))
for i, (k, v) in enumerate(collisions.items()):
    if i >= 15:
        break
    print(' ', k, '=>', sorted(set(v)))

Raw candidates: 492
Unique by normalized name: 492
Potential reductions: 0
Colliding keys: 0


In [5]:
# Diagnose group-aware dedup against V2 merged metadata
import pandas as pd

full_v2_path = '/home/msp25gd/Downloads/res/meta/full_metadata_V2.pkl'
meta_v2 = pd.read_pickle(full_v2_path)

# Build lookup from any known name token -> merged group id
# Use multiple columns because folder names can come from different naming conventions.
def _clean_token(x):
    if x is None:
        return None
    s = str(x).strip().lower()
    if not s:
        return None
    s = s.replace(' ', '').replace('_', '').replace('-', '')
    return s

name_cols = [c for c in ['Reduced', 'Sanitised', 'OBJECT', 'Object'] if c in meta_v2.columns]
lookup = {}
for _, row in meta_v2.iterrows():
    gid = int(row['New Groups'])
    for c in name_cols:
        key = _clean_token(row[c])
        if key is not None and key not in lookup:
            lookup[key] = gid

resolved = 0
unresolved = 0
groups_seen = set()
for raw in cands.tolist():
    key = _clean_token(raw)
    gid = lookup.get(key)
    if gid is None and key is not None and 'normalize_reduced' in globals():
        gid = lookup.get(_clean_token(normalize_reduced(key)))
    if gid is None:
        unresolved += 1
    else:
        resolved += 1
        groups_seen.add(gid)

print('Raw candidates:', len(cands))
print('Resolved to V2 groups:', resolved)
print('Unresolved:', unresolved)
print('Unique V2 groups among resolved candidates:', len(groups_seen))
if resolved > 0:
    print('Potential reduction (resolved only):', resolved - len(groups_seen))

Raw candidates: 492
Resolved to V2 groups: 492
Unresolved: 0
Unique V2 groups among resolved candidates: 492
Potential reduction (resolved only): 0


In [6]:
# Stars filtered out after QuickSearch_V2_Res run.
# "Analyzed sample" is dataset_path (after min_spectra filter).

def _path_to_star_name(p):
    return Path(p).name

sample_stars = [_path_to_star_name(p) for p in dataset_path]

# Final candidate set reported by this run (after optional group dedup).
candidate_stars = [str(x) for x in cands.tolist()]

sample_token_to_name = {}
for name in sample_stars:
    tok = _clean_token(name)
    if tok is not None and tok not in sample_token_to_name:
        sample_token_to_name[tok] = name

candidate_tokens = set()
for raw in candidate_stars:
    tok = _clean_token(raw)
    if tok is not None:
        candidate_tokens.add(tok)
    if tok is not None and 'normalize_reduced' in globals():
        tok2 = _clean_token(normalize_reduced(tok))
        if tok2 is not None:
            candidate_tokens.add(tok2)

filtered_out_stars = [
    name for name in sample_stars
    if _clean_token(name) not in candidate_tokens
]

# Also report stars dropped before quicksearch due to min_spectra.
all_dataset_stars = [_path_to_star_name(p) for p in all_paths]
sample_tokens = {_clean_token(name) for name in sample_stars}
pre_filtered_by_min_spectra = [
    name for name in all_dataset_stars
    if _clean_token(name) not in sample_tokens
]

print('--- Filtering report ---')
print('Total stars in dataset root:', len(all_dataset_stars))
print('Stars analyzed (after min_spectra filter):', len(sample_stars))
print('Final candidates:', len(candidate_stars))
print('Filtered out after quicksearch (non-candidates):', len(filtered_out_stars))
print('Filtered out before quicksearch by min_spectra:', len(pre_filtered_by_min_spectra))

if filtered_out_stars:
    print('\nFirst 20 non-candidates:')
    for s in filtered_out_stars[:20]:
        print(' -', s)

out_dir = Path(args.r)
out_dir.mkdir(parents=True, exist_ok=True)

tag_suffix = args.tag.strip() if args.tag.strip() else 'notag'
non_cand_npy = out_dir / f'filtered_out_non_candidates_{tag_suffix}.npy'
non_cand_txt = out_dir / f'filtered_out_non_candidates_{tag_suffix}.txt'
min_spec_npy = out_dir / f'filtered_out_min_spectra_{tag_suffix}.npy'
min_spec_txt = out_dir / f'filtered_out_min_spectra_{tag_suffix}.txt'

np.save(non_cand_npy, np.array(filtered_out_stars, dtype=object))
np.save(min_spec_npy, np.array(pre_filtered_by_min_spectra, dtype=object))

with open(non_cand_txt, 'w') as f:
    f.write('\n'.join(filtered_out_stars))

with open(min_spec_txt, 'w') as f:
    f.write('\n'.join(pre_filtered_by_min_spectra))

print('\nSaved:')
print(' -', non_cand_npy)
print(' -', non_cand_txt)
print(' -', min_spec_npy)
print(' -', min_spec_txt)

--- Filtering report ---
Total stars in dataset root: 497
Stars analyzed (after min_spectra filter): 497
Final candidates: 492
Filtered out after quicksearch (non-candidates): 5
Filtered out before quicksearch by min_spectra: 0

First 20 non-candidates:
 - hd163745
 - wd0344+073
 - hd22049
 - hd217522
 - he22524225

Saved:
 - /home/msp25gd/ResearchProjectMSc/ResolutionHandling/QuickSearch_V2/filtered_out_non_candidates_V2_RES.npy
 - /home/msp25gd/ResearchProjectMSc/ResolutionHandling/QuickSearch_V2/filtered_out_non_candidates_V2_RES.txt
 - /home/msp25gd/ResearchProjectMSc/ResolutionHandling/QuickSearch_V2/filtered_out_min_spectra_V2_RES.npy
 - /home/msp25gd/ResearchProjectMSc/ResolutionHandling/QuickSearch_V2/filtered_out_min_spectra_V2_RES.txt
